# Recommendation System - SVD-based Node2Vec + Degree-Normalised Combined Embeddings

This notebook combines semantic (BGE-M3) and structural (SVD-based Node2Vec) embeddings
to build a recommendation system for WikiCS articles using cosine similarity.

### Key features:
1. **SVD-based Node2Vec**: Structural embeddings trained via PPMI matrix factorisation
   (pure numpy, no C extensions). Equivalent in spirit to skip-gram Node2Vec.
2. **Power-Law Normalisation**: The Node2Vec component is divided by $\log(\text{degree}+1)^{\alpha}$
   where $\alpha$ is estimated via maximum-likelihood fitting of the degree distribution.
   This penalises high-degree hub nodes that would otherwise dominate recommendations.
3. **Popularity Bias Removal**: The degree normalisation ensures popular nodes are not
   over-represented purely due to their connectivity.
4. **Exact Title Matching**: All test queries use canonical Wikipedia article titles
   (e.g. `Machine_learning`, not `Machine Learning`).
5. **Fuzzy Fallback**: If an exact title is not found, `utils.resolve_title()` finds
   the highest-scoring fuzzy match above a threshold.


In [1]:
import sys
import os
REPO_ROOT = r'C:\programming\github-repos\graph-ending'
utils_parent = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki')
sys.path.insert(0, utils_parent)

import pickle, os, json
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

from utils import (
    load_graph_data,
    fuzzy_search,
    resolve_title,
    load_embeddings,
    build_normalized_embeddings,
    node2vec_svd,
)

DATA_PATH     = "../../data/data-embeddings.json"
BGE_M3_PATH  = "../../cap-embeddings/BAAI_bge-m3/master_embeddings.parquet"
CACHE_DIR    = "../../cache/recommendation-1"
os.makedirs(CACHE_DIR, exist_ok=True)

print("Libraries and utils loaded.")

Libraries and utils loaded.


## 1. Load Graph and Embeddings

In [2]:
nodes, id_to_title, title_to_id, G = load_graph_data(DATA_PATH)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

df_bge = pd.read_parquet(BGE_M3_PATH).sort_values('id').reset_index(drop=True)
print(f"BGE-M3  : {len(df_bge)} nodes, dim={len(df_bge['embedding'].iloc[0])}")

Graph: 11701 nodes, 291039 edges


BGE-M3  : 11701 nodes, dim=1024


## 2. Power-Law Degree Distribution Fit

Estimate the power-law exponent $\alpha$ of the undirected degree distribution
using the `powerlaw` library.  The fitted $\alpha$ is then used to penalise
high-degree nodes in the Node2Vec component:

$$e_{\text{n2v}}^{*} = e_{\text{n2v}} / \log(d+1)^{\alpha}$$


In [3]:
import powerlaw

degrees = [G.degree(n) for n in G.nodes()]

fit = powerlaw.Fit(np.array(degrees, dtype=float), discrete=True, verbose=False)
alpha = fit.power_law.alpha
xmin  = fit.power_law.xmin

print(f"Power-law fit results:")
print(f"  alpha (exponent) = {alpha:.4f}")
print(f"  xmin             = {xmin}")
print(f"  sigma (std err)  = {fit.power_law.sigma:.4f}")

ALPHA_FILE = os.path.join(CACHE_DIR, "alpha.pkl")
with open(ALPHA_FILE, "wb") as f:
    pickle.dump({"alpha": alpha, "xmin": xmin, "degrees": degrees}, f)
print(f"\nAlpha saved to {ALPHA_FILE}")

Power-law fit results:
  alpha (exponent) = 2.9981
  xmin             = 115.0
  sigma (std err)  = 0.0491

Alpha saved to ../../cache/recommendation-1\alpha.pkl


## 3. Node2Vec Training (SVD-based)

Pure-numpy SVD-based Node2Vec (PPMI matrix factorisation). No C extensions required.


In [4]:
# Train SVD-based Node2Vec on the full undirected graph
print("Training SVD-based Node2Vec on full graph...")
G_undirected = G.to_undirected()

n2v_emb = node2vec_svd(
    G_undirected,
    num_walks=10,
    walk_length=80,
    embedding_dim=128,
    seed=42,
)
print(f"  Node2Vec (SVD): {len(n2v_emb)} nodes, dim=128")

# Convert to DataFrame aligned with BGE-M3 IDs
n2v_ids = list(n2v_emb.keys())
df_n2v = pd.DataFrame({
    'id': n2v_ids,
    'embedding': [n2v_emb[k] for k in n2v_ids]
})

print("Building degree-normalised combined embeddings...")
df_combined = build_normalized_embeddings(df_bge, df_n2v, G, alpha)
print(f"Combined embeddings: {len(df_combined)} nodes, dim={len(df_combined['embedding'].iloc[0])}")

EMB_FILE = os.path.join(CACHE_DIR, "combined_embeddings.pkl")
with open(EMB_FILE, "wb") as f:
    pickle.dump(df_combined, f)
print(f"Saved to {EMB_FILE}")


Training SVD-based Node2Vec on full graph...


  Node2Vec (SVD): 11701 nodes, dim=128
Building degree-normalised combined embeddings...


Combined embeddings: 11701 nodes, dim=1152


Saved to ../../cache/recommendation-1\combined_embeddings.pkl


## 4. Precompute Similarity Matrix

In [5]:
embedding_matrix = np.stack(df_combined["embedding"].values)
node_ids_arr = df_combined["id"].values
id_to_idx    = {nid: i for i, nid in enumerate(node_ids_arr)}
idx_to_id    = {i: nid for i, nid in enumerate(node_ids_arr)}
n_nodes      = len(node_ids_arr)

SIM_FILE = os.path.join(CACHE_DIR, "similarity_matrix.pkl")

print("  [comp]  Computing similarity matrix...")
sim_matrix = cosine_similarity(embedding_matrix)
with open(SIM_FILE, "wb") as f:
    pickle.dump(sim_matrix, f)
print(f"  [save]  Saved.")

np.fill_diagonal(sim_matrix, 0.0)

existing_edges = set(G.edges())

def is_linked(src_id, tgt_id):
    return (src_id, tgt_id) in existing_edges

MAPPINGS_FILE = os.path.join(CACHE_DIR, "node_mappings.pkl")
with open(MAPPINGS_FILE, "wb") as f:
    pickle.dump({
        "node_ids_arr": node_ids_arr,
        "id_to_idx":    id_to_idx,
        "idx_to_id":    idx_to_id,
        "id_to_title":  id_to_title,
        "title_to_id":  title_to_id,
    }, f)

print(f"\nReady: {sim_matrix.shape}, {len(existing_edges)} edges")

  [comp]  Computing similarity matrix...


  [save]  Saved.



Ready: (11701, 11701), 291039 edges


## 5. Recommendation Functions

In [6]:
def get_recommendations(query, top_k=10, verbose=True):
    """
    Get top-k recommendations for a query using degree-normalised cosine similarity.
    1. Try exact title match in title_to_id.
    2. Fall back to resolve_title() (fuzzy, best score wins).
    Returns list of dicts: {title, similarity, linked}.
    """
    matched = resolve_title(query, title_to_id)
    if matched is None:
        if verbose:
            print(f"No match found for '{query}'.")
        return None
    if matched != query and verbose:
        print(f"(resolved '{query}' -> '{matched}')")

    node_id = title_to_id[matched]
    idx     = id_to_idx[node_id]
    sims    = sim_matrix[idx]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for i in top_idx:
        rec_id = node_ids_arr[i]
        if rec_id == node_id:
            continue
        results.append({
            "title":      id_to_title[rec_id],
            "similarity": round(float(sims[i]), 6),
            "linked":     is_linked(node_id, rec_id),
        })

    return results[:top_k]


def display_recs(query, top_k=10):
    recs = get_recommendations(query, top_k=top_k)
    if recs is None:
        return
    print(f"\nTop-{top_k} for '{query}':")
    display(pd.DataFrame(recs))
    return recs


## 6. Exact-Match Test Queries

All queries below are exact Wikipedia article titles from the dataset.
They cover broad CS topics to test semantic understanding.

In [7]:
# Vague / broad topic queries (all exact Wikipedia article titles in the graph)
vague_queries = [
    "Natural_language_processing",
    "Operating_system",
    "Programming_language",
    "Database",
    "Cloud_computing",
    "World_Wide_Web",
    "Adversarial_machine_learning",
    "Brownout_(software_engineering)",
    "Computer_and_network_surveillance",
    "Quantum_programming",
]

print(f"Testing {len(vague_queries)} vague/broad topic queries...\n")
for q in vague_queries:
    recs = get_recommendations(q, top_k=5, verbose=False)
    if recs:
        tops = ", ".join([f"{r['title']} ({r['similarity']:.3f})" for r in recs])
        print(f"[VAGUE] {q}")
        print(f"       {tops}")
    else:
        print(f"[NO MATCH] {q}")
    print()


Testing 10 vague/broad topic queries...

[VAGUE] Natural_language_processing
       Machine_translation (0.731), Computational_linguistics (0.707), Natural-language_generation (0.703), Language_model (0.697), Question_answering (0.678)

[VAGUE] Operating_system
       History_of_operating_systems (0.789), Process_(computing) (0.760), Computer_multitasking (0.743), Disk_operating_system (0.737), Real-time_operating_system (0.728)

[VAGUE] Programming_language
       Compiler (0.769), High-level_programming_language (0.755), Comparison_of_programming_languages (0.744), Programming_paradigm (0.731), Syntax_(programming_languages) (0.722)

[VAGUE] Database
       Database_design (0.737), Relational_database (0.718), Federated_database_system (0.716), Database_server (0.705), Database_model (0.702)

[VAGUE] Cloud_computing
       Cloud_storage (0.772), Software_as_a_service (0.735), Cloud_collaboration (0.716), Infrastructure_as_a_service (0.702), Cloud_research (0.688)

[VAGUE] World_Wide_

## 7. Detailed Recommendations

In [8]:
# Detailed recommendations for vague/broad topics
display_recs("Natural_language_processing")
display_recs("Operating_system")
display_recs("Programming_language")
display_recs("Database")
display_recs("Cloud_computing")
display_recs("World_Wide_Web")
print("\n---\n")


Top-10 for 'Natural_language_processing':


,title,similarity,linked
0,Machine_translation,0.730651,True
1,Computational_linguistics,0.707333,True
2,Natural-language_generation,0.703054,True
3,Language_model,0.696888,False
4,Question_answering,0.677896,True
5,Speech_recognition,0.677592,True
6,Association_for_Computational_Linguistics,0.673406,False
7,Parsing,0.671392,True
8,Speech_synthesis,0.655520,True
9,Programming_language,0.641811,False



Top-10 for 'Operating_system':


,title,similarity,linked
0,History_of_operating_systems,0.789462,True
1,Process_(computing),0.759905,True
2,Computer_multitasking,0.742768,True
3,Disk_operating_system,0.736587,True
4,Real-time_operating_system,0.728131,True
5,List_of_operating_systems,0.719761,True
6,MS-DOS,0.718739,True
7,Shell_(computing),0.715239,True
8,List_of_file_systems,0.712971,False
9,Monolithic_kernel,0.711064,True



Top-10 for 'Programming_language':


...


Top-10 for 'Cloud_computing':


,title,similarity,linked
0,Cloud_storage,0.771838,True
1,Software_as_a_service,0.734588,True
2,Cloud_collaboration,0.716401,True
3,Infrastructure_as_a_service,0.702154,True
4,Cloud_research,0.688047,True
5,Amazon_Web_Services,0.669328,True
6,Grid_computing,0.663603,True
7,Utility_computing,0.659891,True
8,Oracle_Cloud,0.653226,True
9,Cloud_computing_issues,0.650008,True



Top-10 for 'World_Wide_Web':


,title,similarity,linked
0,Line_Mode_Browser,0.729031,False
1,Web_server,0.711257,True
2,Linked_data,0.679092,True
3,Libwww,0.668768,False
4,URL,0.667763,True
5,JavaScript,0.665793,True
6,Hypertext_Transfer_Protocol,0.658600,True
7,Semantic_Web,0.648806,True
8,CERN_httpd,0.644621,False
9,Web_service,0.636692,True



---



## 8. Batch Evaluation Table

In [9]:
def get_top_10_table(titles):
    rows = []
    for title in titles:
        recs = get_recommendations(title, top_k=10, verbose=False)
        if recs is None:
            print(f"Warning: '{title}' not found.")
            continue
        row = {"Query": title}
        for i, res in enumerate(recs):
            flag = " [LINK]" if res["linked"] else ""
            row[f"Top {i+1}"] = f"{res['title']} ({res['similarity']:.4f}){flag}"
        rows.append(row)
    return pd.DataFrame(rows)

sample_titles = list(title_to_id.keys())[:20]
display(get_top_10_table(sample_titles))

,Query,Top 1,Top 2,Top 3,Top 4,Top 5,Top 6,Top 7,Top 8,Top 9,Top 10
0,Twilio,Sinch_(company) (0.6022),ServiceNow (0.5919),Zoho_Corporation (0.5867),Content_Guru (0.5866),Mindbody_Inc. (0.5866),Virtustream (0.5865),Tim_Bray (0.5859),"Workday,_Inc. (0.5805)",Cloudreach (0.5695),NetSuite (0.5605)
1,Program_compatibility_date_range,Anticiparallelism (0.6873),Synchronous_Data_Flow (0.6703),Slipstream_(computer_science) (0.6624),Branch_Queue (0.6568),Fast_interrupt_request (0.6463),Hybrid-core_computing (0.6396),Tile_processor (0.6379),International_Symposium_on_Microarchitecture (...,Architectural_state (0.6348),Wide-issue (0.6342)
2,SYSTAT_(DEC),TOPS-10 (0.8038) [LINK],MacvsWindows (0.7708),Commercial_Operating_System_(COS) (0.7688),Berkeley_Timesharing_System (0.7660),TSX-Plus (0.7361),CDC_Kronos (0.7355),4K_Disk_Monitor_System (0.7333),Crash-only_software (0.7286),Kernel_preemption (0.7259),Business_Operating_System_(software) (0.7189)
3,List_of_column-oriented_DBMSes,Information_schema (0.7224),Clustrix (0.6999),Flyway_(software) (0.6933),Vectorwise (0.6912) [LINK],MemSQL (0.6897) [LINK],List_of_in-memory_databases (0.6883),Multi-master_replication (0.6867),Oracle_RAC (0.6858),Virtual_column (0.6856),HSQLDB (0.6832)
4,Stealth_wallpaper,Ipsectrace (0.7431),Michael_Schearer (0.7256),Wi-Fi_deauthentication_attack (0.7065),Columbitech (0.6942),Secure_transmission (0.6853),KARMA_attack (0.6755),Network_encryption_cracking (0.6648),"Fluhrer,_Mantin_and_Shamir_attack (0.6594)",Pre-shared_key (0.6491),Message_forgery (0.6311)
5,Scalable_TCP,TCP_global_synchronization (0.8997),Delay-gradient_congestion_control (0.8562),TCP_window_scale_option (0.8070),TCP_Westwood (0.7972),Tsunami_UDP_Protocol (0.7900),HSTCP (0.7570),Zeta-TCP (0.7558),TCP_Westwood_plus (0.7471),Media_Delivery_Index (0.7437),TCP_delayed_acknowledgment (0.7352)
6,Carrier_IQ,Mobile_cloud_computing (0.7066) [LINK],Digital_omnivore (0.7063),B2G_OS (0.6910),Tizen_Association (0.6868),Modern_Combat:_Sandstorm (0.6538),Qt_Extended (0.6422),Openmoko (0.6414),UIQ (0.6384),GeeksPhone_Revolution (0.6298),Asphalt_6:_Adrenaline (0.6289)
7,ACF2,Resource_Access_Control_Facility (0.7820) [LINK],IBM_System_Management_Facilities (0.7124),TACACS (0.6789) [LINK],Woo–Lam (0.6406) [LINK],Ticket_Granting_Ticket (0.6343),BSD_Authentication (0.6190) [LINK],Generic_Security_Services_Application_Program_...,CRAM-MD5 (0.6143) [LINK],Access_Control_Service (0.6072),Pluggable_authentication_module (0.5975) [LINK]
8,Dorkbot_(malware),Code_Shikara (0.8462),Dropper_(malware) (0.7520),System_Safety_Monitor (0.7138),Facebook_malware (0.7090),Spybot_–_Search_&_Destroy (0.7069),WiperSoft (0.7015),NProtect_GameGuard_Personal_2007 (0.7006),Shedun (0.6970),Greynet (0.6951),Vundo (0.6939)
9,Lout_(software),Flagship_compiler (0.7154),XOS_(mobile_operating_system) (0.7037),Big_V_Telecom (0.7037),McAfee_Institute (0.7037),Workboard (0.7037),Extensible_Threat_Management (0.7037),Tempus_Nova (0.7037),Remark_Media (0.7037),Orange_(database_management_and_monitoring_sof...,LoginRadius (0.7037)
